In [27]:
import rdflib
from rdflib import Graph, URIRef, Literal, Namespace
from rdflib.namespace import RDF, OWL, RDFS
import torch
from torch_geometric.data import HeteroData
from torch_geometric.nn import SAGEConv, HeteroConv
import torch.nn.functional as F
import torch.nn as nn
import torch_geometric.transforms as T
from sentence_transformers import SentenceTransformer

In [28]:
# File paths
ttl_ark = r"C:\Users\yanpe\OneDrive - Metropolia Ammattikorkeakoulu Oy\Research\MD2MV\data\TTL\01ARK\ARK_MET.ttl"
ttl_sensor = r"C:\Users\yanpe\OneDrive - Metropolia Ammattikorkeakoulu Oy\Research\MD2MV\data\TTL\sensors_linked.ttl"

# 1. Load Data
g = rdflib.Graph()
files = [ttl_ark, ttl_sensor] 

for file in files:
    try:
        g.parse(file, format="turtle")
        print(f"Loaded {file}")
    except Exception as e:
        print(f"Error loading {file}: {e}")

# Define Namespaces
BRICK = Namespace("https://brickschema.org/schema/Brick#")
BOT = Namespace("https://w3id.org/bot#")
INST = Namespace("https://lbd.example.com/")
PROPS = Namespace("http://lbd.arch.rwth-aachen.de/props#")

PREFIXES = {
    "brick": BRICK, "bot": BOT, "inst": INST, "rdfs": RDFS, "props": PROPS
}

for p, ns in PREFIXES.items(): g.bind(p, ns)

Loaded C:\Users\yanpe\OneDrive - Metropolia Ammattikorkeakoulu Oy\Research\MD2MV\data\TTL\01ARK\ARK_MET.ttl
Loaded C:\Users\yanpe\OneDrive - Metropolia Ammattikorkeakoulu Oy\Research\MD2MV\data\TTL\sensors_linked.ttl


In [29]:
# Print the first 5 triples to see the raw format
print("--- Raw Triples Sample ---")
for s, p, o in list(g)[:5]:
    print(f"{s}  |  {p}  |  {o}")

# Print all unique Types found
unique_types = set()
for s, p, o in g.triples((None, RDF.type, None)):
    unique_types.add(str(o))
print(f"\n--- Found Node Types ({len(unique_types)}) ---")
print(unique_types)

# Print all unique Predicates found
unique_preds = set()
for s, p, o in g:
    unique_preds.add(str(p))
print(f"\n--- Found Relations ({len(unique_preds)}) ---")
print(unique_preds)


--- Raw Triples Sample ---
https://lbd.example.com/wall_ae67b2bf-8316-4d92-a0e6-1ed84e5b3bb3  |  http://lbd.arch.rwth-aachen.de/props#globalIdIfcRoot_attribute_simple  |  2kPxA$WnPDag3c7jXEMpkp
https://lbd.example.com/wall_1798ccea-aa19-41f7-b372-ad96d543e26c  |  https://w3id.org/bot#hasSubElement  |  https://lbd.example.com/door_fee3f249-9668-4076-ae9d-60d0eae2725e
https://lbd.example.com/railing_81c21791-fb6b-40af-80ed-189e6e37faac  |  http://lbd.arch.rwth-aachen.de/props#objectTypeIfcObject_attribute_simple  |  Railing:KÃ¤sijohde d 30mm
https://lbd.example.com/wall_7c575b5f-b6bb-4516-801b-e3e04fadb854  |  http://www.w3.org/2002/07/owl#sameAs  |  https://lbd.example.com/IfcWallStandardCase_14633015
https://lbd.example.com/flowterminal_9c247209-b941-4d40-af98-a82fda27464b  |  http://lbd.arch.rwth-aachen.de/props#batid_attribute_simple  |  14626612

--- Found Node Types (10) ---
{'https://w3id.org/bot#Site', 'https://w3id.org/bot#Storey', 'https://w3id.org/bot#Building', 'https://brick

In [38]:
from rdflib import RDF
import torch
from torch_geometric.data import HeteroData

# 1. Define the types you want to merge
SENSOR_SUBTYPES = {
    'http://brc/Room_Air_Temperature_Sensor', 
    'http://brc/CO2_Sensor', 
    'http://brc/Humidity_Sensor',
    # Add any other sensor URIs here
}

# 2. Custom Graph Builder
data = HeteroData()
uri_to_id = {}      # Global URI -> Index map (per generic type)
id_to_uri = {}      # Generic Type -> {Index: URI}
node_counts = {}    # Track index for each generic type

# Helper to get generic type
def get_generic_type(rdf_type_uri):
    # Convert rdflib term to string
    type_str = str(rdf_type_uri)
    # Check if it's one of our target sensors
    if type_str in SENSOR_SUBTYPES or "Sensor" in type_str: 
        return "Sensor"
    # Otherwise, keep original type (e.g., 'Space', 'Storey')
    return type_str.split('#')[-1].split('/')[-1]

# --- PASS 1: Identify Nodes & Assign IDs ---
print("Scanning nodes and merging Sensor types...")
for s, p, o in g:
    if p == RDF.type:
        s_str = str(s)
        # Determine if this is a Sensor or something else
        generic_type = get_generic_type(o)
        
        if generic_type not in uri_to_id:
            uri_to_id[generic_type] = {}
            id_to_uri[generic_type] = {}
            node_counts[generic_type] = 0
            
        if s_str not in uri_to_id[generic_type]:
            current_id = node_counts[generic_type]
            uri_to_id[generic_type][s_str] = current_id
            id_to_uri[generic_type][current_id] = s_str
            node_counts[generic_type] += 1

# --- PASS 2: Build Edges with Generic Types ---
print("Building edges for merged types...")
edge_list = {} # Key: (src_type, rel, dst_type), Value: [[src_ids], [dst_ids]]

for s, p, o in g:
    if p == RDF.type: continue
    
    s_str, o_str = str(s), str(o)
    rel_name = str(p).split('#')[-1].split('/')[-1]
    
    # We need to find which Generic Type these URIs belong to
    # This is a bit slow (O(N)), but safe. 
    # In production, you'd store a lookup map: global_uri -> generic_type
    src_type, dst_type = None, None
    
    # Fast lookup if you built a global map, otherwise iterate:
    for g_type, mapping in uri_to_id.items():
        if s_str in mapping: src_type = g_type
        if o_str in mapping: dst_type = g_type
    
    if src_type and dst_type:
        # Construct the consolidated edge key
        edge_key = (src_type, rel_name, dst_type)
        
        # Initialize if new
        if edge_key not in edge_list:
            edge_list[edge_key] = [[], []]
            
        src_id = uri_to_id[src_type][s_str]
        dst_id = uri_to_id[dst_type][o_str]
        
        edge_list[edge_key][0].append(src_id)
        edge_list[edge_key][1].append(dst_id)

# Load into HeteroData
for edge_key, indices in edge_list.items():
    data[edge_key].edge_index = torch.tensor(indices, dtype=torch.long)

print(f"Merged Graph Created. Nodes: {node_counts}")
print(f"Edge Types: {list(data.edge_index_dict.keys())}")

Scanning nodes and merging Sensor types...
Building edges for merged types...
Merged Graph Created. Nodes: {'Element': 51773, 'Space': 1404, 'Sensor': 1775, 'Storey': 12, 'DatatypeProperty': 6, 'ObjectProperty': 1, 'Site': 1, 'Building': 1}
Edge Types: [('Element', 'hasSubElement', 'Element'), ('Storey', 'containsElement', 'Element'), ('Storey', 'hasSpace', 'Space'), ('Space', 'containsElement', 'Element'), ('Sensor', 'isPointOf', 'Space'), ('Building', 'hasStorey', 'Storey'), ('Site', 'hasBuilding', 'Building')]


In [39]:
from sentence_transformers import SentenceTransformer
text_model = SentenceTransformer('all-MiniLM-L6-v2')

# Generate features for the new "Sensor" super-type
# The rdfs:label usually contains "CO2" or "Temperature", so the embedding will capture the difference.
for node_type, mapping in id_to_uri.items():
    labels = []
    for i in range(len(mapping)):
        uri = mapping[i]
        label = g.value(URIRef(uri), RDFS.label)
        if label is None:
            # Fallback: Extract from URI (e.g., "sensor_temp_01")
            label = str(uri).split('/')[-1].replace('_', ' ')
        labels.append(str(label))
        
    embeddings = text_model.encode(labels, convert_to_tensor=True)
    data[node_type].x = embeddings.detach().clone()

In [42]:
class SemanticBuildingGNN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, metadata):
        super().__init__()
        # Projection layer
        self.proj = nn.ModuleDict({
            n_type: nn.Linear(in_channels, hidden_channels) for n_type in metadata[0]
        })
        
        # Conv Layers
        self.conv1 = HeteroConv({
            edge: SAGEConv(hidden_channels, hidden_channels) for edge in metadata[1]
        }, aggr='mean')
        
        self.conv2 = HeteroConv({
            edge: SAGEConv(hidden_channels, out_channels) for edge in metadata[1]
        }, aggr='mean')

    def forward(self, x_dict, edge_index_dict):
        # 1. Initial Projection
        x_dict = {k: self.proj[k](x).relu() for k, x in x_dict.items()}
        
        # 2. Layer 1 with Safe Skip Connection
        out1 = self.conv1(x_dict, edge_index_dict)
        
        new_x_dict = {}
        for k, x in x_dict.items():
            # SAFE ACCESS: use .get(k) instead of [k]
            # If 'DatatypeProperty' is missing from out1, it returns None
            out_val = out1.get(k)
            if out_val is not None:
                new_x_dict[k] = out_val.relu()
            else:
                # If node received no messages, keep original features
                new_x_dict[k] = x
        
        # 3. Layer 2 with Safe Skip Connection
        out2 = self.conv2(new_x_dict, edge_index_dict)
        
        final_x_dict = {}
        for k, x in new_x_dict.items():
            out_val = out2.get(k)
            if out_val is not None:
                final_x_dict[k] = out_val
            else:
                final_x_dict[k] = x
                
        return final_x_dict

In [43]:
import torch_geometric.transforms as T

# 1. Define the single consolidated relation
target_edge = ('Sensor', 'isPointOf', 'Space')

# 2. Make Bidirectional (Sensor <-> Space)
data = T.ToUndirected()(data)

# 3. Split
transform = T.RandomLinkSplit(
    num_val=0.1, 
    num_test=0.1, 
    add_negative_train_samples=True,
    edge_types=[target_edge],
    rev_edge_types=[('Space', 'rev_isPointOf', 'Sensor')]
)
train_data, val_data, test_data = transform(data)

# Move to Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
train_data = train_data.to(device)
data = data.to(device)

# 4. Model (Same as before, but inputs are now unified)
model = SemanticBuildingGNN(
    in_channels=384, 
    hidden_channels=64, 
    out_channels=32, 
    metadata=data.metadata()
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.005)

# 5. Training Loop
print(f"Training on unified relation: {target_edge}")
for epoch in range(1, 151):
    model.train()
    optimizer.zero_grad()
    
    z_dict = model(train_data.x_dict, train_data.edge_index_dict)
    
    # Calculate Loss on the unified edge
    edge_label_index = train_data[target_edge].edge_label_index
    edge_label = train_data[target_edge].edge_label
    
    src, _, dst = target_edge
    out = (z_dict[src][edge_label_index[0]] * z_dict[dst][edge_label_index[1]]).sum(dim=-1)
    loss = F.binary_cross_entropy_with_logits(out, edge_label)
    
    loss.backward()
    optimizer.step()
    
    if epoch % 20 == 0:
        print(f"Epoch {epoch:03d} | Loss: {loss:.4f}")

Training on unified relation: ('Sensor', 'isPointOf', 'Space')
Epoch 020 | Loss: 0.2706
Epoch 040 | Loss: 0.2056
Epoch 060 | Loss: 0.1376
Epoch 080 | Loss: 0.0728
Epoch 100 | Loss: 0.4909
Epoch 120 | Loss: 0.3300
Epoch 140 | Loss: 0.0852


In [44]:
# 6. MEMORY-EFFICIENT PREDICTION
@torch.no_grad()
def predict_relations_efficiently(target_edge_type, top_k=10, batch_size=100):
    model.eval()
    src_type, _, dst_type = target_edge_type
    
    z_dict = model(data.x_dict, data.edge_index_dict)
    src_embs = F.normalize(z_dict[src_type], p=2, dim=-1)
    dst_embs = F.normalize(z_dict[dst_type], p=2, dim=-1)
    
    results = []
    for i in range(0, src_embs.size(0), batch_size):
        chunk = src_embs[i:i + batch_size]
        scores = torch.matmul(chunk, dst_embs.t()) # Cosine similarity matrix
        
        vals, idxs = torch.topk(scores, k=min(top_k, dst_embs.size(0)), dim=-1)
        
        # Transfer results to CPU
        vals, idxs = vals.cpu(), idxs.cpu()
        
        for j in range(chunk.size(0)):
            s_idx = i + j
            for k in range(vals.size(1)):
                results.append((vals[j, k].item(), s_idx, idxs[j, k].item()))

    results.sort(key=lambda x: x[0], reverse=True)
    
    print(f"\n--- Top {top_k} Predictions ---")
    for score, s_idx, d_idx in results[:top_k]:
        print(f"Prob: {score:.4f} | {id_to_uri[src_type][s_idx]} --> {id_to_uri[dst_type][d_idx]}")

In [45]:
# Predict for the unified type
predict_relations_efficiently(('Sensor', 'isPointOf', 'Space'), top_k=20)


--- Top 20 Predictions ---
Prob: 0.8798 | https://lbd.example.com/sensor_4be6edef-dfe9-4f5d-ade7-d4e10c43d7e6 --> https://lbd.example.com/space_809591bb-ccbf-4ff1-9f9a-30f19838b267
Prob: 0.8798 | https://lbd.example.com/sensor_5a7c5841-ca42-40e1-a4a4-8fb3fd4269bc --> https://lbd.example.com/space_809591bb-ccbf-4ff1-9f9a-30f19838b267
Prob: 0.8734 | https://lbd.example.com/sensor_4be6edef-dfe9-4f5d-ade7-d4e10c43d7e6 --> https://lbd.example.com/space_db3ac89f-fee1-4815-92ad-13947ebff9a7
Prob: 0.8734 | https://lbd.example.com/sensor_5a7c5841-ca42-40e1-a4a4-8fb3fd4269bc --> https://lbd.example.com/space_db3ac89f-fee1-4815-92ad-13947ebff9a7
Prob: 0.8664 | https://lbd.example.com/sensor_4be6edef-dfe9-4f5d-ade7-d4e10c43d7e6 --> https://lbd.example.com/space_874157c1-3c08-48b4-a81e-0e314bf5d510
Prob: 0.8664 | https://lbd.example.com/sensor_5a7c5841-ca42-40e1-a4a4-8fb3fd4269bc --> https://lbd.example.com/space_874157c1-3c08-48b4-a81e-0e314bf5d510
Prob: 0.8624 | https://lbd.example.com/sensor_4b

In [37]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from rdflib import Graph, RDFS, URIRef
import rdflib
from sentence_transformers import SentenceTransformer
import torch_geometric.transforms as T
from torch_geometric.data import HeteroData
from torch_geometric.nn import SAGEConv, HeteroConv

# 1. SETUP DEVICE
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# 2. FEATURE GENERATION (With Autograd Fix)
text_model = SentenceTransformer('all-MiniLM-L6-v2')

def extract_semantic_features(g, id_to_uri_dict):
    node_features = {}
    for node_type, mapping in id_to_uri_dict.items():
        print(f"Processing labels for: {node_type}")
        labels = []
        for i in range(len(mapping)):
            uri = mapping[i]
            label = g.value(URIRef(uri), RDFS.label)
            if label is None:
                label = str(uri).split('#')[-1].split('/')[-1].replace('_', ' ')
            labels.append(str(label))
        
        # Clone and detach to escape Inference Mode
        embeddings = text_model.encode(labels, convert_to_tensor=True)
        node_features[node_type] = embeddings.detach().clone()
    return node_features

# Generate features and assign to data
label_x_dict = extract_semantic_features(g, id_to_uri)
for n_type, feat in label_x_dict.items():
    data[n_type].x = feat

# 3. GRAPH PREP & SPLIT
data = T.ToUndirected()(data) # Bi-directional links

target_edge = ('Room_Air_Temperature_Sensor', 'isPointOf', 'Space')
rev_target_edge = ('Space', 'rev_isPointOf', 'Room_Air_Temperature_Sensor')

transform = T.RandomLinkSplit(
    num_val=0.1,
    num_test=0.1,
    disjoint_train_ratio=0.3,
    neg_sampling_ratio=2.0,
    add_negative_train_samples=True,
    edge_types=[target_edge],
    rev_edge_types=[rev_target_edge]
)

train_data, val_data, test_data = transform(data)

# Move all data to GPU/CPU device
train_data = train_data.to(device)
val_data = val_data.to(device)
test_data = test_data.to(device)
data = data.to(device)

# 4. MODEL DEFINITION
class SemanticBuildingGNN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, metadata):
        super().__init__()
        self.proj = nn.ModuleDict({
            n_type: nn.Linear(in_channels, hidden_channels) for n_type in metadata[0]
        })
        self.conv1 = HeteroConv({
            edge: SAGEConv(hidden_channels, hidden_channels) for edge in metadata[1]
        }, aggr='mean')
        self.conv2 = HeteroConv({
            edge: SAGEConv(hidden_channels, out_channels) for edge in metadata[1]
        }, aggr='mean')

    def forward(self, x_dict, edge_index_dict):
        # Project inputs
        x_dict = {k: self.proj[k](x).relu() for k, x in x_dict.items()}
        
        # Layer 1
        out1 = self.conv1(x_dict, edge_index_dict)
        x_dict = {k: (out1[k].relu() if out1[k] is not None else x_dict[k]) for k in x_dict.keys()}
        
        # Layer 2
        out2 = self.conv2(x_dict, edge_index_dict)
        x_dict = {k: (out2[k] if out2[k] is not None else x_dict[k]) for k in x_dict.keys()}
        return x_dict

model = SemanticBuildingGNN(384, 64, 32, train_data.metadata()).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)

# 5. TRAINING LOOP
def train():
    model.train()
    optimizer.zero_grad()
    
    z_dict = model(train_data.x_dict, train_data.edge_index_dict)
    
    edge_label_index = train_data[target_edge].edge_label_index
    edge_label = train_data[target_edge].edge_label
    
    src_type, _, dst_type = target_edge
    src_h = z_dict[src_type][edge_label_index[0]]
    dst_h = z_dict[dst_type][edge_label_index[1]]
    
    # Compute score
    out = (src_h * dst_h).sum(dim=-1)
    loss = F.binary_cross_entropy_with_logits(out, edge_label)
    
    loss.backward()
    optimizer.step()
    return loss.item()

print("\n--- Training Starting ---")
for epoch in range(1, 101):
    loss = train()
    if epoch % 20 == 0:
        print(f"Epoch {epoch:03d} | Loss: {loss:.4f}")

# 6. MEMORY-EFFICIENT PREDICTION
@torch.no_grad()
def predict_relations_efficiently(target_edge_type, top_k=10, batch_size=100):
    model.eval()
    src_type, _, dst_type = target_edge_type
    
    z_dict = model(data.x_dict, data.edge_index_dict)
    src_embs = F.normalize(z_dict[src_type], p=2, dim=-1)
    dst_embs = F.normalize(z_dict[dst_type], p=2, dim=-1)
    
    results = []
    for i in range(0, src_embs.size(0), batch_size):
        chunk = src_embs[i:i + batch_size]
        scores = torch.matmul(chunk, dst_embs.t()) # Cosine similarity matrix
        
        vals, idxs = torch.topk(scores, k=min(top_k, dst_embs.size(0)), dim=-1)
        
        # Transfer results to CPU
        vals, idxs = vals.cpu(), idxs.cpu()
        
        for j in range(chunk.size(0)):
            s_idx = i + j
            for k in range(vals.size(1)):
                results.append((vals[j, k].item(), s_idx, idxs[j, k].item()))

    results.sort(key=lambda x: x[0], reverse=True)
    
    print(f"\n--- Top {top_k} Predictions ---")
    for score, s_idx, d_idx in results[:top_k]:
        print(f"Prob: {score:.4f} | {id_to_uri[src_type][s_idx]} --> {id_to_uri[dst_type][d_idx]}")

predict_relations_efficiently(target_edge)

Using device: cuda
Processing labels for: Element
Processing labels for: Space
Processing labels for: Storey
Processing labels for: Building
Processing labels for: Site
Processing labels for: CO2_Sensor
Processing labels for: Room_Air_Temperature_Sensor
Processing labels for: Humidity_Sensor

--- Training Starting ---
Epoch 020 | Loss: 0.1722
Epoch 040 | Loss: 0.0220
Epoch 060 | Loss: 0.0002
Epoch 080 | Loss: 0.0000
Epoch 100 | Loss: 0.0000

--- Top 10 Predictions ---
Prob: -0.0432 | https://lbd.example.com/sensor_773c322d-c0a3-4bc0-bc1c-f4b8249f1a5f --> https://lbd.example.com/space_9a2d747d-8135-4a8b-a46e-05ad7cc74cd1
Prob: -0.0432 | https://lbd.example.com/sensor_8a6e104d-3e41-45d6-a8c8-5884cb82195e --> https://lbd.example.com/space_9a2d747d-8135-4a8b-a46e-05ad7cc74cd1
Prob: -0.0439 | https://lbd.example.com/sensor_773c322d-c0a3-4bc0-bc1c-f4b8249f1a5f --> https://lbd.example.com/space_9a2d747d-8135-4a8b-a46e-05ad7cc74cf3
Prob: -0.0439 | https://lbd.example.com/sensor_8a6e104d-3e41-4

In [31]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [32]:
from rdflib import RDFS
from sentence_transformers import SentenceTransformer
import torch

# Load a lightweight, high-performance model
text_model = SentenceTransformer('all-MiniLM-L6-v2')

def extract_label_features(g, id_to_uri, dim=384):
    node_features = {}
    for node_type, mapping in id_to_uri.items():
        print(f"Processing labels for: {node_type}")
        labels = []
        for i in range(len(mapping)):
            uri = mapping[i]
            # Get rdfs:label, fallback to URI fragment if missing
            label = g.value(rdflib.URIRef(uri), RDFS.label)
            if label is None:
                label = str(uri).split('#')[-1].split('/')[-1].replace('_', ' ')
            labels.append(str(label))
        
        # Batch encode labels into 384-dim vectors
        embeddings = text_model.encode(labels, convert_to_tensor=True, show_progress_bar=True)
        node_features[node_type] = embeddings
    return node_features

# Apply to your data
label_x = extract_label_features(g, id_to_uri)
for n_type, feat in label_x.items():
    data[n_type].x = feat

Processing labels for: Element


Batches:   0%|          | 0/1618 [00:00<?, ?it/s]

Processing labels for: Space


Batches:   0%|          | 0/44 [00:00<?, ?it/s]

Processing labels for: Storey


...

Batches:   0%|          | 0/20 [00:00<?, ?it/s]

Processing labels for: Room_Air_Temperature_Sensor


Batches:   0%|          | 0/36 [00:00<?, ?it/s]

Processing labels for: Humidity_Sensor


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

In [33]:
import torch_geometric.transforms as T

# 1. Make the graph bidirectional (very important for BOT context)
data = T.ToUndirected()(data)

# 2. Split for Link Prediction
target_edge = ('Room_Air_Temperature_Sensor', 'isPointOf', 'Space')

transform = T.RandomLinkSplit(
    num_val=0.1,
    num_test=0.1,
    disjoint_train_ratio=0.3,
    neg_sampling_ratio=2.0,
    add_negative_train_samples=True,
    edge_types=[target_edge],
    rev_edge_types=[('Space', 'rev_isPointOf', 'Room_Air_Temperature_Sensor')]
)

train_data, val_data, test_data = transform(data)
train_data = train_data.to(device)
val_data = val_data.to(device)
test_data = test_data.to(device)

In [34]:
from torch_geometric.nn import SAGEConv, HeteroConv
import torch.nn as nn

class SemanticBuildingGNN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, metadata):
        super().__init__()
        # Project 384-dim text to 64-dim hidden space
        self.proj = nn.ModuleDict({
            n_type: nn.Linear(in_channels, hidden_channels) for n_type in metadata[0]
        })
        
        self.conv1 = HeteroConv({
            edge: SAGEConv(hidden_channels, hidden_channels) for edge in metadata[1]
        }, aggr='mean')
        
        self.conv2 = HeteroConv({
            edge: SAGEConv(hidden_channels, out_channels) for edge in metadata[1]
        }, aggr='mean')

    def forward(self, x_dict, edge_index_dict):
        # Initial projection
        x_dict = {k: self.proj[k](x).relu() for k, x in x_dict.items()}
        
        # Layer 1 + Skip Connection
        out1 = self.conv1(x_dict, edge_index_dict)
        x_dict = {k: (out1[k].relu() if out1[k] is not None else x_dict[k]) for k in x_dict.keys()}
        
        # Layer 2 + Skip Connection
        out2 = self.conv2(x_dict, edge_index_dict)
        x_dict = {k: (out2[k] if out2[k] is not None else x_dict[k]) for k in x_dict.keys()}
        
        return x_dict

model = SemanticBuildingGNN(384, 64, 32, train_data.metadata()).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)

In [35]:
@torch.no_grad()
def predict_relations_efficiently(target_edge_type, top_k=10, batch_size=100):
    model.eval()
    src_type, _, dst_type = target_edge_type
    
    # Get all embeddings
    z_dict = model(data.x_dict, data.edge_index_dict)
    src_embs = F.normalize(z_dict[src_type], p=2, dim=-1) # [N_sensors, 32]
    dst_embs = F.normalize(z_dict[dst_type], p=2, dim=-1) # [N_spaces, 32]
    
    results = []
    
    # Process sensors in chunks to save memory
    for i in range(0, src_embs.size(0), batch_size):
        chunk = src_embs[i:i + batch_size]
        # Similarity score for this batch of sensors against ALL spaces
        scores = torch.matmul(chunk, dst_embs.t()) # [batch, N_spaces]
        
        # Get top candidates for each sensor in this chunk
        vals, idxs = torch.topk(scores, k=min(top_k, dst_embs.size(0)), dim=-1)
        
        for j in range(chunk.size(0)):
            sensor_idx = i + j
            for k in range(vals.size(1)):
                results.append((vals[j, k].item(), sensor_idx, idxs[j, k].item()))

    # Sort all found candidates and pick the absolute top_k
    results.sort(key=lambda x: x[0], reverse=True)
    
    print(f"\n--- Top {top_k} Semantic Predictions ---")
    for score, s_idx, d_idx in results[:top_k]:
        print(f"Score: {score:.4f} | {id_to_uri[src_type][s_idx]} --> {id_to_uri[dst_type][d_idx]}")

# After training for ~100 epochs...
predict_relations_efficiently(target_edge)

RuntimeError: Expected all tensors to be on the same device, but got index is on cpu, different from other tensors on cuda:0 (when checking argument in method wrapper_CUDA__index_select)

In [15]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import HeteroData
from torch_geometric.nn import SAGEConv, HeteroConv
import torch_geometric.transforms as T

# ==========================================
# 1. MODEL DEFINITION (Manual HeteroGNN)
# ==========================================
class ManualHeteroGNN(torch.nn.Module):
    def __init__(self, hidden_channels, out_channels, metadata):
        super().__init__()
        # Layer 1
        self.conv1 = HeteroConv({
            edge_type: SAGEConv((-1, -1), hidden_channels)
            for edge_type in metadata[1]
        }, aggr='sum')
        # Layer 2
        self.conv2 = HeteroConv({
            edge_type: SAGEConv((-1, -1), out_channels)
            for edge_type in metadata[1]
        }, aggr='sum')

    def forward(self, x_dict, edge_index_dict):
        # Layer 1 with Skip Connection for isolated nodes
        out_dict1 = self.conv1(x_dict, edge_index_dict)
        x_dict_mid = {}
        for node_type, x in x_dict.items():
            if node_type in out_dict1 and out_dict1[node_type] is not None:
                x_dict_mid[node_type] = out_dict1[node_type].relu()
            else:
                # If node is isolated, pass through original features
                x_dict_mid[node_type] = x

        # Layer 2
        out_dict2 = self.conv2(x_dict_mid, edge_index_dict)
        final_x_dict = {}
        for node_type, x in x_dict_mid.items():
            if node_type in out_dict2 and out_dict2[node_type] is not None:
                final_x_dict[node_type] = out_dict2[node_type]
            else:
                final_x_dict[node_type] = x
        return final_x_dict

# ==========================================
# 2. PREPARATION & DATA SPLIT
# ==========================================
# target_edge should be the one you want to predict
target_edge = ('Room_Air_Temperature_Sensor', 'isPointOf', 'Space')

# Ensure all nodes have features (using random as fallback, or degree)
for node_type in data.node_types:
    if not hasattr(data[node_type], 'x'):
        data[node_type].x = torch.randn(data[node_type].num_nodes, 16)

# Split data for link prediction
transform = T.RandomLinkSplit(
    num_val=0.1,
    num_test=0.1,
    disjoint_train_ratio=0.3, # Uses some edges for supervision, some for message passing
    neg_sampling_ratio=2.0,
    add_negative_train_samples=True,
    edge_types=[target_edge],
)
train_data, val_data, test_data = transform(data)

# Initialize Model
model = ManualHeteroGNN(hidden_channels=32, out_channels=16, metadata=data.metadata())
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# ==========================================
# 3. TRAINING LOOP
# ==========================================
def train():
    model.train()
    optimizer.zero_grad()
    
    # Forward pass
    z_dict = model(train_data.x_dict, train_data.edge_index_dict)
    
    # Get labels and edge indices for the target relation
    edge_label_index = train_data[target_edge].edge_label_index
    edge_label = train_data[target_edge].edge_label
    
    # Dot product prediction
    src_type, _, dst_type = target_edge
    src_h = z_dict[src_type][edge_label_index[0]]
    dst_h = z_dict[dst_type][edge_label_index[1]]
    
    out = (src_h * dst_h).sum(dim=-1)
    loss = F.binary_cross_entropy_with_logits(out, edge_label)
    
    loss.backward()
    optimizer.step()
    return loss.item()

print("Starting Training...")
for epoch in range(1, 101):
    loss = train()
    if epoch % 20 == 0:
        print(f"Epoch {epoch:03d} | Loss: {loss:.4f}")

# ==========================================
# 4. PREDICTION (Finding Missing Relations)
# ==========================================
@torch.no_grad()
def find_missing_relations(target_edge_type, top_k=10):
    model.eval()
    src_type, rel, dst_type = target_edge_type
    
    # Get embeddings for the whole graph
    embeddings = model(data.x_dict, data.edge_index_dict)
    
    src_emb = embeddings[src_type] 
    dst_emb = embeddings[dst_type]
    
    # Calculate probability matrix
    # [Num Sensors] x [Num Spaces]
    score_matrix = torch.matmul(src_emb, dst_emb.t()).sigmoid()
    
    # Mask existing edges (don't predict what we already know)
    if target_edge_type in data.edge_index_dict:
        existing = data[target_edge_type].edge_index
        score_matrix[existing[0], existing[1]] = 0
    
    # Get top candidates
    values, indices = torch.topk(score_matrix.flatten(), top_k)
    
    print(f"\n--- Top {top_k} Suggested Relations for {rel} ---")
    for v, idx in zip(values, indices):
        s_idx = int(idx // score_matrix.size(1))
        d_idx = int(idx % score_matrix.size(1))
        
        src_uri = id_to_uri[src_type][s_idx]
        dst_uri = id_to_uri[dst_type][d_idx]
        
        print(f"Prob: {v:.4f} | {src_uri} -> {dst_uri}")

# Run prediction
find_missing_relations(target_edge)

C:\Users\yanpe\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\torch_geometric\nn\conv\hetero_conv.py:76: UserWarning: There exist node types ({'Room_Air_Temperature_Sensor', 'Humidity_Sensor', 'Site', 'CO2_Sensor'}) whose representations do not get updated during message passing as they do not occur as destination type in any edge type. This may lead to unexpected behavior.
  warnings.warn(


Starting Training...
Epoch 020 | Loss: 0.0441
Epoch 040 | Loss: 0.0035
Epoch 060 | Loss: 0.0013
Epoch 080 | Loss: 0.0008
Epoch 100 | Loss: 0.0006

--- Top 10 Suggested Relations for isPointOf ---
Prob: 1.0000 | https://lbd.example.com/sensor_0053893f-e20e-4d49-95cb-3372f8c66bb3 -> https://lbd.example.com/space_9f2d4c94-88cd-4f44-a1e2-3428bd162477
Prob: 1.0000 | https://lbd.example.com/sensor_0053893f-e20e-4d49-95cb-3372f8c66bb3 -> https://lbd.example.com/space_6db44793-2631-417e-914a-b29f9b786c62
Prob: 1.0000 | https://lbd.example.com/sensor_0053893f-e20e-4d49-95cb-3372f8c66bb3 -> https://lbd.example.com/space_b5585a10-8222-4139-bc99-e21934c2939a
Prob: 1.0000 | https://lbd.example.com/sensor_0053893f-e20e-4d49-95cb-3372f8c66bb3 -> https://lbd.example.com/space_f37e1d2d-4dd0-4b61-91b3-c11f0adf17ee
Prob: 1.0000 | https://lbd.example.com/sensor_0053893f-e20e-4d49-95cb-3372f8c66bb3 -> https://lbd.example.com/space_64ca89e7-dc68-4c95-be9f-53df4469f0d0
Prob: 1.0000 | https://lbd.example.com/

In [5]:
class ManualHeteroGNN(torch.nn.Module):
    def __init__(self, hidden_channels, out_channels, metadata):
        super().__init__()
        self.conv1 = HeteroConv({
            edge_type: SAGEConv((-1, -1), hidden_channels)
            for edge_type in metadata[1]
        }, aggr='sum')
        
        self.conv2 = HeteroConv({
            edge_type: SAGEConv((-1, -1), out_channels)
            for edge_type in metadata[1]
        }, aggr='sum')

    def forward(self, x_dict, edge_index_dict):
        # 1. First Convolution
        out_dict = self.conv1(x_dict, edge_index_dict)
        
        # 2. SAFETY: Filter out None values and apply ReLU
        # Some node types might not receive messages and return None
        x_dict = {
            key: x.relu() for key, x in out_dict.items() 
            if x is not None
        }
        
        # 3. Second Convolution
        out_dict = self.conv2(x_dict, edge_index_dict)
        
        # 4. SAFETY: Filter out None values again
        x_dict = {
            key: x for key, x in out_dict.items() 
            if x is not None
        }
        
        return x_dict

# Initialize the model with the data's metadata
# data.metadata() returns ([node_types], [edge_types])
model = ManualHeteroGNN(hidden_channels=32, out_channels=16, metadata=data.metadata())

C:\Users\yanpe\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\torch_geometric\nn\conv\hetero_conv.py:76: UserWarning: There exist node types ({'Room_Air_Temperature_Sensor', 'Humidity_Sensor', 'Site', 'CO2_Sensor'}) whose representations do not get updated during message passing as they do not occur as destination type in any edge type. This may lead to unexpected behavior.
  warnings.warn(


In [7]:
import torch_geometric.transforms as T

# 1. Define which relation we want to predict
# Using the one from your previous error
target_edge = ('Room_Air_Temperature_Sensor', 'isPointOf', 'Space')

# 2. Setup the split
# num_val=0.1 (10% for validation), num_test=0.1 (10% for testing)
# This will automatically hide some edges from the 'train' graph 
# so the model can't "cheat" by seeing them during training.
transform = T.RandomLinkSplit(
    num_val=0.1,
    num_test=0.2,
    key="edge_label", # This creates a 'labels' attribute for training
    add_negative_train_samples=True, 
    edge_types=[target_edge], # We only split the target relation
    rev_edge_types=None 
)

train_data, val_data, test_data = transform(data)

print(f"Original edges: {data[target_edge].edge_index.size(1)}")
print(f"Train edges: {train_data[target_edge].edge_index.size(1)}")

Original edges: 827
Train edges: 580


In [9]:


class LinkPredictor(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        self.lin = nn.Sequential(
            nn.Linear(in_channels * 2, in_channels),
            nn.ReLU(),
            nn.Linear(in_channels, 1)
        )

    def forward(self, z_src, z_dst):
        # Combine sensor embedding and space embedding
        combined = torch.cat([z_src, z_dst], dim=-1)
        return self.lin(combined)

# Initialize
predictor = LinkPredictor(in_channels=16)
optimizer = torch.optim.Adam(list(model.parameters()) + list(predictor.parameters()), lr=0.01)

def train():
    model.train()
    predictor.train()
    optimizer.zero_grad()

    # 1. Get embeddings using the TRAINING edges only
    # We use a try-except here to catch any persistent PyG internal errors 
    # during the message passing phase.
    try:
        z_dict = model(train_data.x_dict, train_data.edge_index_dict)
    except AttributeError:
        # Fallback: if message passing fails because a node is too isolated, 
        # we use the raw features (x) as the embedding.
        z_dict = train_data.x_dict 

    # 2. Get the specific edges for this training batch
    edge_label_index = train_data[target_edge].edge_label_index
    labels = train_data[target_edge].edge_label

    # 3. Predict
    src_type, _, dst_type = target_edge
    z_src = z_dict[src_type][edge_label_index[0]]
    z_dst = z_dict[dst_type][edge_label_index[1]]
    
    predictions = predictor(z_src, z_dst).squeeze()
    
    loss = torch.nn.functional.binary_cross_entropy_with_logits(predictions, labels)
    loss.backward()
    optimizer.step()
    return loss.item()

In [13]:
def forward(self, x_dict, edge_index_dict):
    # Layer 1
    out_dict = self.conv1(x_dict, edge_index_dict)
    
    # Check every node type in the original input
    new_x_dict = {}
    for node_type, x in x_dict.items():
        if node_type in out_dict and out_dict[node_type] is not None:
            # If node received messages, use them
            new_x_dict[node_type] = out_dict[node_type].relu()
        else:
            # If node is isolated, keep its original features (crucial for BOT!)
            # We use a linear layer or just pass through to keep dimensions consistent
            new_x_dict[node_type] = x 

    # Layer 2
    out_dict2 = self.conv2(new_x_dict, edge_index_dict)
    
    final_dict = {}
    for node_type, x in new_x_dict.items():
        if node_type in out_dict2 and out_dict2[node_type] is not None:
            final_dict[node_type] = out_dict2[node_type]
        else:
            final_dict[node_type] = x
            
    return final_dict

In [14]:
@torch.no_grad()
def find_missing_relations(target_edge_type, top_k=10):
    model.eval()
    src_type, rel, dst_type = target_edge_type
    
    # Use the full data object for final inference
    embeddings = model(data.x_dict, data.edge_index_dict)
    
    # Check if our target types actually exist in the output
    if src_type not in embeddings or dst_type not in embeddings:
        print(f"Error: One of the types {src_type} or {dst_type} has no embeddings.")
        return

    src_emb = embeddings[src_type] 
    dst_emb = embeddings[dst_type]
    
    # Memory safety for your 55k nodes: 
    # Instead of one massive matrix, let's do it in chunks if necessary
    # But for now, let's stick to the matrix if your RAM allows:
    score_matrix = torch.matmul(src_emb, dst_emb.t()).sigmoid()
    
    # Masking
    if target_edge_type in data.edge_index_dict:
        existing_edges = data[target_edge_type].edge_index
        score_matrix[existing_edges[0], existing_edges[1]] = 0
    
    values, indices = torch.topk(score_matrix.flatten(), top_k)
    
    # ... (rest of your print logic)

In [ ]:
@torch.no_grad()
def find_missing_relations(target_edge_type, top_k=10):
    model.eval()
    src_type, rel, dst_type = target_edge_type
    
    # Use the full data object for final inference
    embeddings = model(data.x_dict, data.edge_index_dict)
    
    # Check if our target types actually exist in the output
    if src_type not in embeddings or dst_type not in embeddings:
        print(f"Error: One of the types {src_type} or {dst_type} has no embeddings.")
        return

    src_emb = embeddings[src_type] 
    dst_emb = embeddings[dst_type]
    
    # Memory safety for your 55k nodes: 
    # Instead of one massive matrix, let's do it in chunks if necessary
    # But for now, let's stick to the matrix if your RAM allows:
    score_matrix = torch.matmul(src_emb, dst_emb.t()).sigmoid()
    
    # Masking
    if target_edge_type in data.edge_index_dict:
        existing_edges = data[target_edge_type].edge_index
        score_matrix[existing_edges[0], existing_edges[1]] = 0
    
    values, indices = torch.topk(score_matrix.flatten(), top_k)
    
    # ... (rest of your print logic)

AttributeError: 'NoneType' object has no attribute 'dim'